# Notebook 04: Cost Optimization Pattern

**CCA Pattern:** Prompt Caching vs Batch API misuse

This notebook demonstrates the correct cost optimization strategy for live customer support. We show:

1. **Anti-Pattern**: Why the Batch API is ALWAYS wrong for live support (conceptual)
2. **Correct Pattern**: Prompt Caching with `cache_control` for 90% savings on repeated context
3. **Compare**: Token accounting difference between cached and uncached runs

## Setup

Install dependencies and set up services. Requires `ANTHROPIC_API_KEY` environment variable.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve()))

import anthropic
from helpers import compare_results, print_usage

from customer_service.agent.agent_loop import run_agent_loop
from customer_service.agent.system_prompts import (
    POLICY_DOCUMENT,
    get_system_prompt,
    get_system_prompt_with_caching,
)
from customer_service.data.customers import CUSTOMERS
from customer_service.data.scenarios import SCENARIOS
from customer_service.services.audit_log import AuditLog
from customer_service.services.container import ServiceContainer
from customer_service.services.customer_db import CustomerDatabase
from customer_service.services.escalation_queue import EscalationQueue
from customer_service.services.financial_system import FinancialSystem
from customer_service.services.policy_engine import PolicyEngine

In [ ]:
def make_services() -> ServiceContainer:
    return ServiceContainer(
        customer_db=CustomerDatabase(CUSTOMERS),
        policy_engine=PolicyEngine(),
        financial_system=FinancialSystem(),
        escalation_queue=EscalationQueue(),
        audit_log=AuditLog(),
    )


client = anthropic.Anthropic()

## Anti-Pattern: Batch API for Live Customer Support

<div style="background-color:#fff0f0; border-left:4px solid #cc0000; padding:16px; margin:12px 0; border-radius:4px;">

**WRONG APPROACH: Using Batch API to cut costs**

A common misconception is that using the Batch API reduces costs by 50% for customer support operations. This is **always wrong** for live support.

</div>

### Why Batch API Is Wrong for Live Support

**1. Latency** — The Batch API processes requests asynchronously with up to **24-hour** turnaround time. A customer waiting for a refund resolution cannot wait 24 hours.

**2. Not ZDR-eligible** — The Batch API is not eligible for Zero Data Retention (ZDR). Organizations with strict data security requirements often require ZDR for customer data. The Real-Time API supports ZDR; the Batch API does not.

**3. Wrong cost lever** — The Batch API saves costs by accepting delay. For live support, the correct cost lever is **Prompt Caching**: sending the same large static context on every API call provides up to **90% cost savings** on repeated tokens — far better than the Batch API's 50% discount — with no latency penalty.

> **CCA Exam Tip:** Use this decision rule:
> 
> - **Someone waiting?** → Real-Time API + Prompt Caching
> - **No one waiting + no ZDR requirement?** → Batch API acceptable
> 
> **"Batch API for live support"** = ALWAYS WRONG. Eliminate immediately on the exam.

The module `customer_service.anti_patterns.batch_api_live` contains the full teaching explanation. There is no runnable code for the Batch API anti-pattern because it should never be executed for live support.

## Correct Pattern: Prompt Caching with `cache_control`

<div style="background-color:#f0fff0; border-left:4px solid #00aa00; padding:16px; margin:12px 0; border-radius:4px;">

**CORRECT APPROACH: Real-Time API + Prompt Caching**

Mark large static context blocks (policy documents, tool descriptions) with `cache_control: {"type": "ephemeral"}`. After the first call writes to cache, subsequent calls read from cache at **10% of the normal input token cost**.

</div>

We will compare **three runs**:

- **Run 1**: Uncached — plain string system prompt, no `cache_control`
- **Run 2**: Cached Turn 1 — first call with `cache_control` (writes to cache)
- **Run 3**: Cached Turn 2 — second call (reads from cache at 10% cost)

Look for `cache_creation_input_tokens` (Run 2) and `cache_read_input_tokens` (Run 3) in the output.

In [ ]:
print(f"POLICY_DOCUMENT token estimate: {len(POLICY_DOCUMENT) // 4}")
print("  (Minimum for caching on claude-sonnet-4-6: 2048 tokens)")
print()
print(f"Uncached format: plain string ({type(get_system_prompt()).__name__})")
print(f"Cached format: list-of-blocks ({type(get_system_prompt_with_caching()).__name__})")
print(f"Blocks: {len(get_system_prompt_with_caching())}")
print(f"Block 1 has cache_control: {'cache_control' in get_system_prompt_with_caching()[1]}")

### Run 1: Uncached (baseline)

Plain string system prompt — no `cache_control`, no caching. Every token in the policy document is charged at the full input rate.

In [ ]:
class _UsageWrapper:
    pass


scenario = SCENARIOS["happy_path"]
services_uncached = make_services()
result_uncached = run_agent_loop(
    client,
    services_uncached,
    scenario["message"],
    get_system_prompt(),  # plain string — no caching
)
w = _UsageWrapper()
w.usage = result_uncached.usage
print_usage(w)
print(f"\nCache read tokens:  {result_uncached.usage.cache_read_input_tokens}")
print(f"Cache write tokens: {result_uncached.usage.cache_creation_input_tokens}")

### Run 2: Cached — Turn 1 (cache write)

First call with `cache_control` on the POLICY_DOCUMENT block. The SDK writes the policy block to cache — you will see `cache_creation_input_tokens > 0`. This first call costs **1.25x** the normal input rate for the cached portion (cache write overhead).

In [ ]:
services_cached_1 = make_services()
result_cached_1 = run_agent_loop(
    client,
    services_cached_1,
    scenario["message"],
    get_system_prompt_with_caching(),  # list-of-blocks with cache_control
)
w = _UsageWrapper()
w.usage = result_cached_1.usage
print_usage(w)
print(f"\nCache write tokens: {result_cached_1.usage.cache_creation_input_tokens}")
print(f"Cache read tokens:  {result_cached_1.usage.cache_read_input_tokens}")

### Run 3: Cached — Turn 2 (cache read)

Second call immediately after Run 2. The policy block is now in cache. You will see `cache_read_input_tokens > 0` — those tokens are charged at **10% of the normal input rate** (90% savings on the cached portion).

In [ ]:
services_cached_2 = make_services()
result_cached_2 = run_agent_loop(
    client,
    services_cached_2,
    scenario["message"],
    get_system_prompt_with_caching(),
)
w = _UsageWrapper()
w.usage = result_cached_2.usage
print_usage(w)
print(f"\nCache write tokens: {result_cached_2.usage.cache_creation_input_tokens}")
print(f"Cache read tokens:  {result_cached_2.usage.cache_read_input_tokens}")

## Compare Results

Comparing uncached (Run 1) vs cached second call (Run 3). The `cache_read` tokens in Run 3 were charged at 10% cost. The `Delta` column shows the percentage change — negative delta on input costs is the goal.

In [ ]:
compare_results(
    {
        "input_tokens": result_uncached.usage.input_tokens,
        "output_tokens": result_uncached.usage.output_tokens,
        "cache_read": result_uncached.usage.cache_read_input_tokens,
        "cache_write": result_uncached.usage.cache_creation_input_tokens,
    },
    {
        "input_tokens": result_cached_2.usage.input_tokens,
        "output_tokens": result_cached_2.usage.output_tokens,
        "cache_read": result_cached_2.usage.cache_read_input_tokens,
        "cache_write": result_cached_2.usage.cache_creation_input_tokens,
    },
)

## CCA Exam Tip: Prompt Caching Economics

> **CCA Exam Tip:**
> 
> **Prompt Caching** is the correct cost optimization for live customer support:
> 
> - **Cache write** (first call): 1.25x input cost — slight overhead to populate cache
> - **Cache read** (subsequent calls): 0.10x input cost — **90% savings** on cached tokens
> - Break-even: After just 2 calls, caching is cheaper than no caching
> - Works with: Large static policy documents, tool schemas, knowledge bases
> 
> **Key rule:** Mark the LAST static block with `cache_control`. Blocks before the cache > breakpoint are all cached together. Don't put `cache_control` on small dynamic blocks.
> 
> **Batch API for live support** → ALWAYS WRONG. It has up to 24-hour latency and is > not ZDR-eligible. Real-Time API + Prompt Caching is the answer.

## Summary

| Pattern | Approach | Latency | Cost | ZDR |
|---------|----------|---------|------|-----|
| Anti-Pattern | Batch API | up to 24 hours | 50% discount | NOT eligible |
| Correct | Real-Time + Prompt Caching | sub-second | 90% savings on cached tokens | Eligible |

**Key files:**

- `src/customer_service/agent/system_prompts.py` — `get_system_prompt_with_caching()`, `POLICY_DOCUMENT`
- `src/customer_service/anti_patterns/batch_api_live.py` — `BATCH_API_EXPLANATION` teaching constant
- `notebooks/helpers.py` — `print_usage()` with cache token display

**CCA Rule:** 'Someone waiting? → Real-Time API + Prompt Caching. Batch API is ALWAYS wrong for live support.'